In [1]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [2]:
import sys
sys.path.append("../")

In [3]:
from mbt_gym.agents.BaselineAgents import CarteaJaimungalMmAgent
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.wrappers import *
from mbt_gym.rewards.RewardFunctions import PnL, CjMmCriterion
from mbt_gym.stochastic_processes.midprice_models import BrownianMotionMidpriceModel
from mbt_gym.stochastic_processes.arrival_models import PoissonArrivalModel
from mbt_gym.stochastic_processes.fill_probability_models import ExponentialFillFunction
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics

### Create market making environment

In [4]:
import sys
sys.path.append("../") # This version of the notebook is in the subfolder "notebooks" of the repo

import gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import quad

from copy import deepcopy


from mbt_gym.agents.BaselineAgents import *
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *
import torch
#print(torch.cuda.is_available())
#print(torch.cuda.get_device_name())
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
seed = 42


## Varying fad proportion (paramter q)

### Parameters

In [5]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

In [6]:
fill_exponent = 0

In [7]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t) + 0.5 * (c**2) * v(t))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each fad proportion
psi_values = {q: compute_psi(q) for q in fads_proportion_values}

print("psi values:", psi_values)

psi values: {0.0: 15.0, 0.2: 14.985756598048642, 0.4: 14.943105475282236, 0.6: 14.872283180889763, 0.8: 14.773681637922692, 1: 14.647844682743013}


In [8]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float=0.6, psi:float = 15.0):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories, seed=seed)
    arrival_model = FadsInformedUniformedTradersArrivalModel(phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                terminal_time=terminal_time,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    # arrival_model = ModifiedPoissonArrivalModel(phi=phi,
    #                                             psi=psi,
    #                                             gamma=gamma,
    #                                             fads_proportion=fads_proportion,
    #                                             sigma=sigma,
    #                                             step_size=terminal_time/n_steps,
    #                                             num_trajectories=num_trajectories,
    #                                             seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent =fill_exponent,
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories, seed=seed)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [9]:
results_dict = {}
for q, psi_val in zip(fads_proportion_values, psi_values.values()):
    vec_env = get_as_env(
        num_trajectories=1000,
        fads_proportion=q,
        psi=psi_val
    )

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )

    results_dict[(q, psi_val)] = dict(
        results=results,
        rewards=total_rewards,
        obs=observations
    )


--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 15.0
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.0
------------------------------------

Precomputed A(T) value: -0.000999999999999994
Precomputed A(0) value: -0.04624809852934575
✓ Pre-computed ODE solutions for 1000 time points

--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 14.985756598048642
  Fill Exponent (k): 1
  Volatility (

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")



--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 14.647844682743013
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 1
------------------------------------

Precomputed A(T) value: -0.0009999999999999968
Precomputed A(0) value: -0.04648818717353375
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [10]:
header = f"{'Fads Prop':>10} | {'Psi':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for (fads_prop, psi_val), result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10.2f} | {psi_val:10.4f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15.2f} | {std_inv:13.2f}")


 Fads Prop |        Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-----------------------------------------------------------------------------------
      0.00 |    15.0000 |     215.84 |      14.92 |           -0.07 |          2.32
      0.20 |    14.9858 |     218.30 |      27.80 |           -0.08 |          2.29
      0.40 |    14.9431 |     227.45 |      50.71 |           -0.07 |          2.31
      0.60 |    14.8723 |     242.40 |      75.85 |           -0.08 |          2.30
      0.80 |    14.7737 |     264.28 |     103.29 |           -0.09 |          2.31
      1.00 |    14.6478 |     294.10 |     133.45 |           -0.09 |          2.32


## Varying eta

### Parameters

In [11]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0

fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta_values = [2.5, 5, 7.5, 10.0, 12.5]
phi = 15
k = 1
gamma = k
alpha=0.001
mu=0
big_phi=0.1

In [12]:

# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each eta
psi_values = {eta: compute_psi(fads_proportion, eta) for eta in eta_values}

# pretty print
for eta, val in psi_values.items():
    print(f"eta={eta:.1f} -> psi={val:.6f}")

eta=2.5 -> psi=14.572885
eta=5.0 -> psi=14.758861
eta=7.5 -> psi=14.832907
eta=10.0 -> psi=14.872283
eta=12.5 -> psi=14.896670


In [13]:
def get_as_env(num_trajectories:int = 1,eta:float=10, psi:float=15.0):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories, seed=seed)
    arrival_model = FadsInformedUniformedTradersArrivalModel(phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                terminal_time=terminal_time,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    # arrival_model = ModifiedPoissonArrivalModel(phi=phi,
    #                                             psi=psi,
    #                                             gamma=gamma,
    #                                             fads_proportion=fads_proportion,
    #                                             sigma=sigma,
    #                                             step_size=terminal_time/n_steps,
    #                                             num_trajectories=num_trajectories,
    #                                             seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent =fill_exponent,
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories, seed=seed)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [14]:
results_dict = {}
for eta in eta_values:
    vec_env = get_as_env(num_trajectories=1000, eta=eta, psi=psi_values[eta])

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[eta] = dict(results=results, rewards=total_rewards, obs=observations)


--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 2.5
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 14.572885423499939
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.0010000000000000015
Precomputed A(0) value: -0.04653973792355661
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


KeyboardInterrupt: 

In [ ]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |     143.44 |     159.60 |          -0.094 | 2.3300566516718
         5 |     234.91 |     108.99 |          -0.087 | 2.3007457486649847
       7.5 |     244.59 |      89.48 |          -0.074 | 2.307926341978877
      10.0 |     242.40 |      75.85 |          -0.079 | 2.3045084074483215
      12.5 |     238.75 |      65.87 |          -0.077 | 2.3063111238512466


## Varying gamma parameter

### Parameters

In [ ]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion_values = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma_values = [0, 1, 2, 3]
alpha=0.001
mu=0
big_phi=0.1

In [ ]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta, gamma):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each gamma
psi_values = {gamma: compute_psi(fads_proportion, eta, gamma) for gamma in gamma_values}

# pretty print
for gamma, val in psi_values.items():
    print(f"gamma={gamma} -> psi={val:.6f}")

gamma=0 -> psi=15.000000
gamma=1 -> psi=14.872283
gamma=2 -> psi=14.495463
gamma=3 -> psi=13.888033


In [ ]:
def get_as_env(num_trajectories:int = 1,gamma:float=1, psi:float=15.0):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories, seed=seed)
    arrival_model = FadsInformedUniformedTradersArrivalModel(phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                terminal_time=terminal_time,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    # arrival_model = ModifiedPoissonArrivalModel(phi=phi,
    #                                             psi=psi,
    #                                             gamma=gamma,
    #                                             fads_proportion=fads_proportion,
    #                                             sigma=sigma,
    #                                             step_size=terminal_time/n_steps,
    #                                             num_trajectories=num_trajectories,
    #                                             seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent =fill_exponent,
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories, seed=seed)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [ ]:
results_dict = {}
for gamma in gamma_values:
    vec_env = get_as_env(num_trajectories=1000, gamma=gamma, psi=psi_values[gamma])

    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)

    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)

    results_dict[gamma] = dict(results=results, rewards=total_rewards, obs=observations)


--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 0
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 15.0
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.000999999999999994
Precomputed A(0) value: -0.04624809852934575
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")



--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 14.872283180889763
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.0010000000000000059
Precomputed A(0) value: -0.04633477530151822
✓ Pre-computed ODE solutions for 1000 time points

--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 2
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 14.495463275588145
  Fill Exponent (k): 1

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")



--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 3
  Order Flow Sensitivity (phi): 15
  Fads Order Flow Sensitivity (psi): 13.888033107804624
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.0010000000000000002
Precomputed A(0) value: -0.04701814389507355
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [ ]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |     311.34 |      97.52 |          -0.039 | 2.4518317642122183
         1 |     242.40 |      75.85 |          -0.079 | 2.3045084074483215
         2 |     174.21 |      75.71 |            -0.1 | 2.337520053389917
         3 |     109.95 |      92.43 |          -0.103 | 2.510854635378162


## Varying Informed trader proportion (psi and phi)

### Parameters

In [ ]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion = 0.6

# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

In [ ]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(phi):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t) + 0.5 * (c**2) * v(t))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

def phi_from_informed(perc_informed):
    """
    Given the percentage of informed traders (0-100),
    compute phi according to the paper.
    """
    return 30 * (1 - perc_informed / 100.0)
def psi_from_informed(perc_informed):
    """
    Given percentage of informed traders, compute phi and psi.
    """
    phi = phi_from_informed(perc_informed)
    psi = compute_psi(phi)  # your eq (61) implementation

    return phi, psi

In [ ]:
# Build the dictionary
def build_phi_psi_dict(percentages):
    results = {}
    for perc in percentages:
        phi, psi = psi_from_informed(perc)
        results[perc] = {"phi": phi, "psi": psi}
    return results

# Example usage
percentages = [0, 25, 50, 75, 100]
phi_psi_dict = build_phi_psi_dict(percentages)
for perc, vals in phi_psi_dict.items():
    print(f"{perc}% informed → phi = {vals['phi']:.2f}, psi = {vals['psi']:.4f}")


0% informed → phi = 30.00, psi = 0.0000
25% informed → phi = 22.50, psi = 7.3239
50% informed → phi = 15.00, psi = 14.6478
75% informed → phi = 7.50, psi = 21.9718
100% informed → phi = 0.00, psi = 29.2957


In [ ]:
def get_as_env(num_trajectories:int = 1, phi:float=15, psi:float=15):
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories, seed=seed)
    arrival_model = FadsInformedUniformedTradersArrivalModel(phi=phi,
                                                psi=psi,
                                                k=k,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                terminal_time=terminal_time,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    # arrival_model = ModifiedPoissonArrivalModel(phi=phi,
    #                                             psi=psi,
    #                                             gamma=gamma,
    #                                             fads_proportion=fads_proportion,
    #                                             sigma=sigma,
    #                                             step_size=terminal_time/n_steps,
    #                                             num_trajectories=num_trajectories,
    #                                             seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent =fill_exponent,
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories, seed=seed)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=n_steps,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

In [ ]:
results_dict = {}
for perc, vals in phi_psi_dict.items():
    phi, psi = vals['phi'], vals['psi']
    
    # Set up environment and agent
    vec_env = get_as_env(num_trajectories=1000, phi=phi, psi=psi)
    vec_as = OptimizedFullInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)
    
    # Generate trajectory and results
    observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(vec_env=vec_env, agent=vec_as)
    
    # Store in results dictionary keyed by percentage
    results_dict[perc] = {
        "results": results,
        "rewards": total_rewards,
        "obs": observations
    }


--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 30.0
  Fads Order Flow Sensitivity (psi): 0.0
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.000999999999999994
Precomputed A(0) value: -0.04624809852934575
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")



--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 22.5
  Fads Order Flow Sensitivity (psi): 7.323922341371507
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.0009999999999999968
Precomputed A(0) value: -0.046367713428987746
✓ Pre-computed ODE solutions for 1000 time points

--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 15.0
  Fads Order Flow Sensitivity (psi): 14.647844682743013
  Fill Exponent (k

/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")



--- Model Parameter Verification ---
  Terminal Time (T): 1.0
  Step Size (dt): 0.001
  Per-step Inventory Aversion (big_phi): 0.1
  Terminal Inventory Aversion (alpha): 0.001
  Midprice Drift (mu): 0
  Fads Mean Reversion (eta): 10.0
  Order Flow Intensity (gamma): 1
  Order Flow Sensitivity (phi): 0.0
  Fads Order Flow Sensitivity (psi): 29.295689365486027
  Fill Exponent (k): 1
  Volatility (sigma): 1.0
  Fads Proportion (q): 0.6
------------------------------------

Precomputed A(T) value: -0.000999999999999997
Precomputed A(0) value: -0.04673175031910744
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/../mbt_gym/agents/BaselineAgents.py:526: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [ ]:
header = f"{'Perc':>6} | {'Phi':>8} | {'Psi':>8} | {'Mean Spread':>11} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for perc, result in results_dict.items():
    phi, psi = phi_psi_dict[perc]['phi'], phi_psi_dict[perc]['psi']
    mean_spread = result['results'].loc['Inventory', 'Mean spread']
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    
    print(f"{perc:6}% | {phi:8.2f} | {psi:8.4f} | {mean_spread:11.5f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

  Perc |      Phi |      Psi | Mean Spread |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
------------------------------------------------------------------------------------------------------
     0% |    30.00 |   0.0000 |     2.09241 |     311.34 |      97.52 |          -0.039 | 2.4518317642122183
    25% |    22.50 |   7.3239 |     2.09264 |     277.31 |      84.30 |          -0.077 | 2.3462887716562086
    50% |    15.00 |  14.6478 |     2.09289 |     241.05 |      76.23 |          -0.066 | 2.303398358947058
    75% |     7.50 |  21.9718 |     2.09313 |     205.61 |      73.13 |          -0.077 | 2.2836529946557116
   100% |     0.00 |  29.2957 |     2.09337 |     170.22 |      77.10 |          -0.085 | 2.3263222046827474
